# Congested Traffic Policy V2

Revised congested-traffic training run using a less impossible curriculum-style distribution, faster policy decisions, explicit TTC observation, stronger collision/safety penalties, and longer training. This notebook writes to separate `congested_traffic_policy_v2` artifacts so the original 20k experiments remain intact.


In [ ]:
from __future__ import annotations

import importlib
import json
import os
import sys
from copy import deepcopy
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
        nested = candidate / "highway-rl-decision-making"
        if (nested / "src").exists() and (nested / "notebooks").exists():
            return nested
    raise RuntimeError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
HELPER_DIR = PROJECT_ROOT / "notebooks" / "_shared"
helper_dir_str = str(HELPER_DIR)
if helper_dir_str not in sys.path:
    sys.path.insert(0, helper_dir_str)

import dqn_notebook_utils as dqn_utils
dqn_utils = importlib.reload(dqn_utils)

adaptive_metric_columns = dqn_utils.adaptive_metric_columns
build_dqn_args = dqn_utils.build_dqn_args
build_env_config = dqn_utils.build_env_config
evaluate_saved_model = dqn_utils.evaluate_saved_model
load_dqn_backend = dqn_utils.load_dqn_backend
train_and_display = dqn_utils.train_and_display

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "congested_traffic_policy"
RESULTS_SUBDIR = "congested_traffic_policy_v2"
RESULTS_DIR = PROJECT_ROOT / "artifacts" / "dqn" / RESULTS_SUBDIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Notebook dir:", NOTEBOOK_DIR)
print("Results dir:", RESULTS_DIR)

## Shared Congested Traffic Configuration

In [ ]:
# Revised congested traffic setup for all seven experiments.
# V2 is intentionally a curriculum-style step down from the original stress test:
# fewer vehicles, larger ego spacing, faster policy decisions, TTC observation, and stronger safety costs.
environment_profile = "structured_baseline"
eval_environment_profile = environment_profile

# Change traffic, episode, observation, and TTC-observation settings here.
# DQN, SAC, saved-model evaluation, diagnostics, and long SAC rendering all derive from this block.
CONGESTED_ENV_CONFIG_INFO = {
    "lanes_count": 3,
    "vehicles_count": 30,
    "duration": 40,
    "ego_spacing": 1.8,
    "vehicles_density": 1.2,
    "simulation_frequency": 15,
    "policy_frequency": 3,
    "other_vehicles_type": "highway_env.vehicle.behavior.IDMVehicle",
    "initial_lane_id": None,
    "offroad_terminal": False,
    "observed_vehicles_count": 12,
    "observation_features": ["presence", "x", "y", "vx", "vy"],
    "observation_absolute": False,
    "observation_see_behind": True,
    "ttc_observation_enabled": True,
    "ttc_feature_name": "front_ttc",
    "ttc_cap": 10.0,
    "ttc_gap_cap": 120.0,
    "ttc_lane_y_threshold": 0.50,
    "ttc_front_only": True,
    "ttc_normalize": True,
    "ttc_include_lane_context": True,
    "ttc_lane_context_sides": ["left", "right"],
}

_ENVIRONMENT_OVERRIDE_KEYS = [
    "lanes_count",
    "vehicles_count",
    "duration",
    "ego_spacing",
    "vehicles_density",
    "simulation_frequency",
    "policy_frequency",
    "other_vehicles_type",
    "initial_lane_id",
    "offroad_terminal",
]

congested_environment_overrides = {
    key: CONGESTED_ENV_CONFIG_INFO[key]
    for key in _ENVIRONMENT_OVERRIDE_KEYS
}
congested_eval_environment_overrides = dict(congested_environment_overrides)

congested_observation_config = {
    "vehicles_count": CONGESTED_ENV_CONFIG_INFO["observed_vehicles_count"],
    "features": list(CONGESTED_ENV_CONFIG_INFO["observation_features"]),
    "absolute": CONGESTED_ENV_CONFIG_INFO["observation_absolute"],
    "see_behind": CONGESTED_ENV_CONFIG_INFO["observation_see_behind"],
}
congested_ttc_observation_config = {
    "enabled": CONGESTED_ENV_CONFIG_INFO["ttc_observation_enabled"],
    "feature_name": CONGESTED_ENV_CONFIG_INFO["ttc_feature_name"],
    "ttc_cap": CONGESTED_ENV_CONFIG_INFO["ttc_cap"],
    "gap_cap": CONGESTED_ENV_CONFIG_INFO["ttc_gap_cap"],
    "lane_y_threshold": CONGESTED_ENV_CONFIG_INFO["ttc_lane_y_threshold"],
    "front_only": CONGESTED_ENV_CONFIG_INFO["ttc_front_only"],
    "normalize": CONGESTED_ENV_CONFIG_INFO["ttc_normalize"],
    "include_lane_context": CONGESTED_ENV_CONFIG_INFO["ttc_include_lane_context"],
    "lane_context_sides": list(CONGESTED_ENV_CONFIG_INFO["ttc_lane_context_sides"]),
}
action_config = {
    "type": "DiscreteMetaAction",
}
base_reward_config = {
    "collision_reward": -5.0,
    "right_lane_reward": 0.03,
    "high_speed_reward": 0.25,
    "lane_change_reward": -0.02,
    "normalize_reward": False,
}
speed_config = {
    "reward_speed_range": [15.0, 28.0],
}

# Continuous surrounding-driver behavior: 0 blue/conservative, 100 red/aggressive.
congested_driver_aggressiveness_config = {
    "enabled": True,
    "distribution": "uniform",
    "min_score": 0.0,
    "max_score": 100.0,
    "fixed_score": None,
    "normal_mean": 50.0,
    "normal_std": 20.0,
    "conservative": {
        "target_speed": 18.0,
        "acc_max": 4.0,
        "comfort_acc_max": 2.0,
        "comfort_acc_min": -4.0,
        "delta": 4.5,
        "time_wanted": 2.4,
        "distance_wanted": 14.0,
        "politeness": 0.8,
        "lane_change_min_acc_gain": 0.8,
        "lane_change_max_braking_imposed": 1.0,
        "lane_change_delay": 1.5,
    },
    "aggressive": {
        "target_speed": 30.0,
        "acc_max": 7.0,
        "comfort_acc_max": 5.5,
        "comfort_acc_min": -6.5,
        "delta": 3.5,
        "time_wanted": 0.6,
        "distance_wanted": 6.0,
        "politeness": 0.0,
        "lane_change_min_acc_gain": 0.05,
        "lane_change_max_braking_imposed": 3.5,
        "lane_change_delay": 0.5,
    },
}

# Optional per-observed-vehicle state feature for the separate aggressiveness-aware DQN study.
# Adds one normalized column to the Kinematics/TTC observation: 0.0 conservative, 1.0 aggressive.
congested_driver_aggressiveness_observation_config = {
    "enabled": True,
    "feature_name": "driver_aggressiveness",
    "normalize": True,
    "ego_value": 0.0,
    "missing_value": 0.0,
}

safety_ttc_flow_reward_config = {
    "enabled": True,
    "ttc_safe_threshold": 4.0,
    "ttc_target": 6.0,
    "ttc_cap": 10.0,
    "low_ttc_penalty_weight": 2.0,
    "max_low_ttc_penalty": 2.0,
    "safe_ttc_bonus_weight": 0.05,
    "max_safe_ttc_bonus": 0.10,
    "lag_penalty_weight": 0.08,
    "speed_tolerance": 2.0,
    "max_lag_penalty": 0.5,
    "rear_ttc_pressure": 5.0,
    "rear_pressure_floor": 0.15,
    "flow_radius": 120.0,
    "lanes": "ego_and_adjacent",
}

lane_change_safety_config = {
    "enabled": True,
    "ttc_cap": 10.0,
    "gap_cap": 120.0,
    "front_ttc_safe": 4.0,
    "rear_ttc_safe": 4.0,
    "front_gap_safe": 12.0,
    "rear_gap_safe": 12.0,
    "penalty_weight": 1.25,
    "max_penalty": 1.5,
    "penalty_power": 1.0,
    "unavailable_lane_penalty": 1.0,
    "penalize_unavailable_lane": True,
    "use_ttc": True,
    "use_gap": True,
}

adaptive_wide_band_config = {
    "enabled": True,
    "mode": "ttc_safety_override",
    "ttc_midpoint": 4.0,
    "ttc_temperature": 0.75,
    "ttc_cap": 10.0,
    "safety_ttc_threshold": 2.5,
    "unsafe_action_penalty": 1.0,
    "min_target_speed": 12.0,
    "max_target_speed": 30.0,
    "faster_max_delta": 6.0,
    "slower_min_delta": 1.0,
    "slower_max_delta": 8.0,
    "cruise_speed": 24.0,
    "action_speed_delta": 2.0,
}

attention_config = {
    "features_dim": 128,
    "attention_heads": 2,
    "attention_dropout": 0.0,
    "presence_feature_idx": 0,
    "embedding_arch": "128,128",
    "net_arch": "128,128",
}

timesteps = 100_000
saved_model_eval_episodes = 1000
eval_episodes_during_training = 20
n_envs = min(4, os.cpu_count() or 1)
seed = 42
train_freq = 4
gradient_steps = 4
saved_model_eval_seed = seed + 10_000
saved_model_eval_name = f"saved_model_eval_{saved_model_eval_episodes}_episodes"

hyperparameters = {
    "learning_rate": 1e-4,
    "buffer_size": 200_000,
    "learning_starts": 10_000,
    "batch_size": 128,
    "gamma": 0.99,
    "target_update_interval": 2_000,
    "train_freq": train_freq,
    "gradient_steps": gradient_steps,
    "exploration_fraction": 0.40,
    "exploration_initial_eps": 1.0,
    "exploration_final_eps": 0.05,
    "progress_every": 5_000,
    "verbose": 0,
}

base_env_config = build_env_config(
    profile_name=environment_profile,
    profile_overrides=congested_environment_overrides,
    observation=congested_observation_config,
    action=action_config,
    reward=base_reward_config,
    speed=speed_config,
    adaptive_longitudinal={"enabled": False},
    rear_flow={"enabled": False},
    traffic_flow_reward={"enabled": False},
    safety_ttc_flow_reward={"enabled": False},
    driver_aggressiveness=congested_driver_aggressiveness_config,
    ttc_observation=congested_ttc_observation_config,
    lane_change_safety=lane_change_safety_config,
)

display(pd.DataFrame({"editable_env_config_info": pd.Series(CONGESTED_ENV_CONFIG_INFO)}))
display(pd.DataFrame({"derived_congested_base_env_v2": pd.Series(base_env_config)}))


## Runner Helpers

In [ ]:
completed_experiments: dict[str, dict] = {}


def make_congested_env_config(
    *,
    safety_reward: bool = False,
    adaptive: bool = False,
    aggressiveness_state: bool = False,
    lane_change_safety: bool = True,
) -> dict:
    return build_env_config(
        profile_name=environment_profile,
        profile_overrides=congested_environment_overrides,
        observation=congested_observation_config,
        action=action_config,
        reward=base_reward_config,
        speed=speed_config,
        adaptive_longitudinal=adaptive_wide_band_config if adaptive else {"enabled": False},
        rear_flow={"enabled": False},
        traffic_flow_reward={"enabled": False},
        safety_ttc_flow_reward=safety_ttc_flow_reward_config if safety_reward else {"enabled": False},
        lane_change_safety=lane_change_safety_config if lane_change_safety else {"enabled": False},
        driver_aggressiveness=congested_driver_aggressiveness_config,
        driver_aggressiveness_observation=(
            congested_driver_aggressiveness_observation_config
            if aggressiveness_state
            else {"enabled": False}
        ),
        ttc_observation=congested_ttc_observation_config,
    )


def run_congested_experiment(
    *,
    run_name: str,
    label: str,
    backend_module: str,
    safety_reward: bool = False,
    adaptive: bool = False,
    attention: bool = False,
    aggressiveness_state: bool = False,
    lane_change_safety: bool = True,
    seed_offset: int = 0,
) -> dict:
    trainer, _, _, results_dir, default_device = load_dqn_backend(
        backend_module=backend_module,
        notebook_subdir="congested_traffic_policy",
        results_subdir=RESULTS_SUBDIR,
    )
    env_config = make_congested_env_config(
        safety_reward=safety_reward,
        adaptive=adaptive,
        aggressiveness_state=aggressiveness_state,
        lane_change_safety=lane_change_safety,
    )
    args = build_dqn_args(
        results_dir=results_dir,
        run_name=run_name,
        timesteps=timesteps,
        eval_episodes=eval_episodes_during_training,
        seed=seed + seed_offset,
        num_envs=n_envs,
        device=default_device,
        hyperparameters=hyperparameters,
        extra=attention_config if attention else None,
    )

    display(pd.DataFrame({"training_and_eval_env": pd.Series(env_config)}))
    summary = train_and_display(
        trainer,
        args,
        env_config,
        label=label,
    )
    saved_eval_summary = evaluate_saved_model(
        trainer,
        summary_path=results_dir / run_name / "summary.json",
        env_config=env_config,
        episodes=saved_model_eval_episodes,
        seed=saved_model_eval_seed + seed_offset,
        name=saved_model_eval_name,
        label=label,
    )
    output = {
        "run_name": run_name,
        "label": label,
        "backend_module": backend_module,
        "safety_reward": bool(safety_reward),
        "adaptive": bool(adaptive),
        "attention": bool(attention),
        "aggressiveness_state": bool(aggressiveness_state),
        "lane_change_safety": bool(lane_change_safety),
        "summary": summary,
        "saved_eval_summary": saved_eval_summary,
    }
    completed_experiments[run_name] = output
    return output

## Experiment 1: Baseline DQN + Lane Context/Safety

In [ ]:
experiment_1_lane_safety_baseline_dqn = run_congested_experiment(
    run_name="congested_lane_safety_baseline_dqn_v2_100k",
    label="Congested Traffic - Baseline DQN + Lane Context/Safety",
    backend_module="elurant_dqn",
    seed_offset=0,
)

## Experiment 2: Attention DQN + Lane Context/Safety

In [ ]:
experiment_2_lane_safety_attention_dqn = run_congested_experiment(
    run_name="congested_lane_safety_attention_dqn_v2_100k",
    label="Congested Traffic - Attention DQN + Lane Context/Safety",
    backend_module="attention_dqn",
    attention=True,
    seed_offset=100,
)

## Experiment 3: Attention DQN + Lane Context/Safety + TTC Reward Safety

In [ ]:
experiment_3_lane_safety_attention_reward = run_congested_experiment(
    run_name="congested_lane_safety_attention_safety_reward_v2_100k",
    label="Congested Traffic - Attention DQN + Lane Context/Safety + TTC Reward Safety",
    backend_module="attention_dqn",
    attention=True,
    safety_reward=True,
    seed_offset=200,
)

## Experiment 4: Adaptive Attention DQN + Lane Context/Safety + TTC Reward Safety

In [ ]:
experiment_4_lane_safety_adaptive_attention_reward = run_congested_experiment(
    run_name="congested_lane_safety_adaptive_attention_safety_reward_v2_100k",
    label="Congested Traffic - Adaptive Attention DQN + Lane Context/Safety + TTC Reward Safety",
    backend_module="attention_dqn",
    attention=True,
    safety_reward=True,
    adaptive=True,
    seed_offset=300,
)

## Previous Experiment 5 Skipped

In [ ]:
print("Skipping previous v2 experiment 5; lane-safety sequence now uses four staged DQN runs.")

## Previous Experiment 6 Skipped

In [ ]:
print("Skipping previous v2 experiment 6; lane-safety sequence now uses four staged DQN runs.")

## Previous Experiment 7 Skipped

In [ ]:
print("Skipping previous v2 experiment 7; lane-safety sequence now uses four staged DQN runs.")

## Previous Experiment 8 Skipped


In [ ]:
print("Skipping previous v2 experiment 8; lane-safety sequence now uses four staged DQN runs.")


## Experiment 9: Hierarchical DQN Lane Command + Keep-Lane Speed Policy


In [ ]:
import time

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
from gymnasium import spaces
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback, CallbackList, EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

DQN_SCRIPT_DIR = PROJECT_ROOT / "src" / "deep_learning" / "DQN"
dqn_script_dir_str = str(DQN_SCRIPT_DIR)
if dqn_script_dir_str not in sys.path:
    sys.path.insert(0, dqn_script_dir_str)

from adaptive_longitudinal import compute_same_lane_ttc, make_highway_env_with_adaptive_longitudinal

HRL_RUN_NAME = "congested_hrl_lane_command_keep_lane_speed_v2_100k"
HRL_LABEL = "Congested Traffic - Hierarchical DQN Lane Command + Keep-Lane Speed Policy"
HRL_SPEED_TIMESTEPS = timesteps
HRL_UPPER_TIMESTEPS = timesteps
HRL_EVAL_EPISODES = saved_model_eval_episodes
HRL_FORCE_RETRAIN = False
HRL_N_ENVS = n_envs
HRL_TTC_CAP = 10.0
HRL_TTC_SAFE = 4.0
HRL_LOCAL_TRAFFIC_RADIUS = 120.0
HRL_SPEED_ACTION_LABELS = ["SLOWER", "IDLE", "FASTER"]
HRL_UPPER_ACTION_LABELS = ["LANE_LEFT", "LANE_RIGHT", "KEEP_LANE"]

HRL_HYPERPARAMETERS = {
    "learning_rate": hyperparameters["learning_rate"],
    "buffer_size": hyperparameters["buffer_size"],
    "learning_starts": hyperparameters["learning_starts"],
    "batch_size": hyperparameters["batch_size"],
    "gamma": hyperparameters["gamma"],
    "target_update_interval": hyperparameters["target_update_interval"],
    "train_freq": hyperparameters["train_freq"],
    "gradient_steps": hyperparameters["gradient_steps"],
    "exploration_fraction": hyperparameters["exploration_fraction"],
    "exploration_initial_eps": hyperparameters["exploration_initial_eps"],
    "exploration_final_eps": hyperparameters["exploration_final_eps"],
    "verbose": 0,
}


def hrl_base_env_config() -> dict:
    config = make_congested_env_config(
        safety_reward=False,
        adaptive=False,
        aggressiveness_state=False,
        lane_change_safety=False,
    )
    # Existing HRL speed/upper policies were trained on Kinematics + front_ttc: (12, 6).
    # Keep the explicit HRL lane-context logic below, but do not append TTC lane-context columns.
    config["ttc_observation"] = {**config.get("ttc_observation", {}), "include_lane_context": False}
    return config


def hrl_make_base_env(config: dict, render_mode=None):
    return make_highway_env_with_adaptive_longitudinal(render_mode=render_mode, config=config)


def hrl_action_index(env, label: str) -> int:
    return int(env.unwrapped.action_type.actions_indexes[label])


def hrl_forward_speed(vehicle) -> float:
    return float(vehicle.speed * np.cos(getattr(vehicle, "heading", 0.0)))


def hrl_speed_score(speed: float, config: dict) -> float:
    low, high = config.get("reward_speed_range", [15.0, 28.0])
    return float(np.clip((float(speed) - float(low)) / max(float(high) - float(low), 1e-6), 0.0, 1.0))


def hrl_lane_index_from_delta(env, lane_delta: int):
    vehicle = env.unwrapped.vehicle
    lane_index = getattr(vehicle, "lane_index", None) or getattr(vehicle, "target_lane_index", None)
    if lane_index is None:
        return None, False
    lane_from, lane_to, lane_id = lane_index
    lane_count = len(env.unwrapped.road.network.graph[lane_from][lane_to])
    candidate_id = int(lane_id) + int(lane_delta)
    valid = 0 <= candidate_id < lane_count
    candidate_id = int(np.clip(candidate_id, 0, lane_count - 1))
    return (lane_from, lane_to, candidate_id), bool(valid)


def hrl_lane_context(env, lane_delta: int = 0) -> dict:
    unwrapped = env.unwrapped
    ego = unwrapped.vehicle
    road = unwrapped.road
    lane_index, valid = hrl_lane_index_from_delta(env, lane_delta)
    if lane_index is None or road is None:
        return {
            "valid": False,
            "front_ttc": HRL_TTC_CAP,
            "rear_ttc": HRL_TTC_CAP,
            "front_gap": HRL_LOCAL_TRAFFIC_RADIUS,
            "rear_gap": HRL_LOCAL_TRAFFIC_RADIUS,
            "lane_mean_speed": hrl_forward_speed(ego),
        }

    lane = road.network.get_lane(lane_index)
    ego_s, _ = lane.local_coordinates(ego.position)
    ego_v = hrl_forward_speed(ego)
    front_gap = np.inf
    rear_gap = np.inf
    front_ttc = HRL_TTC_CAP
    rear_ttc = HRL_TTC_CAP
    local_speeds = []

    for other in road.vehicles:
        if other is ego or getattr(other, "lane_index", None) != lane_index:
            continue
        other_s, _ = lane.local_coordinates(other.position)
        distance = float(other_s - ego_s)
        if abs(distance) <= HRL_LOCAL_TRAFFIC_RADIUS:
            local_speeds.append(hrl_forward_speed(other))
        if distance > 0.0 and distance < front_gap:
            front_gap = float(distance)
            closing = max(0.0, ego_v - hrl_forward_speed(other))
            front_ttc = HRL_TTC_CAP if closing <= 1e-6 else float(np.clip(front_gap / closing, 0.0, HRL_TTC_CAP))
        elif distance < 0.0 and abs(distance) < rear_gap:
            rear_gap = float(abs(distance))
            closing = max(0.0, hrl_forward_speed(other) - ego_v)
            rear_ttc = HRL_TTC_CAP if closing <= 1e-6 else float(np.clip(rear_gap / closing, 0.0, HRL_TTC_CAP))

    if not np.isfinite(front_gap):
        front_gap = HRL_LOCAL_TRAFFIC_RADIUS
    if not np.isfinite(rear_gap):
        rear_gap = HRL_LOCAL_TRAFFIC_RADIUS
    lane_mean_speed = float(np.mean(local_speeds)) if local_speeds else ego_v
    return {
        "valid": bool(valid),
        "front_ttc": float(front_ttc),
        "rear_ttc": float(rear_ttc),
        "front_gap": float(front_gap),
        "rear_gap": float(rear_gap),
        "lane_mean_speed": lane_mean_speed,
    }


def hrl_local_traffic_speed(env) -> float:
    ego = env.unwrapped.vehicle
    ego_x = float(ego.position[0])
    speeds = [
        hrl_forward_speed(other)
        for other in env.unwrapped.road.vehicles
        if other is not ego and abs(float(other.position[0]) - ego_x) <= HRL_LOCAL_TRAFFIC_RADIUS
    ]
    return float(np.mean(speeds)) if speeds else hrl_forward_speed(ego)


def hrl_ttc_low_penalty(front_ttc: float) -> float:
    return float(np.clip((HRL_TTC_SAFE - float(front_ttc)) / HRL_TTC_SAFE, 0.0, 1.0) ** 2)


class HRLKeepLaneSpeedEnv(gym.Wrapper):
    """Low-level speed option. Lateral command is fixed to keep-lane/IDLE."""

    def __init__(self, env: gym.Env, config: dict):
        super().__init__(env)
        self.config = dict(config)
        self.action_space = spaces.Discrete(len(HRL_SPEED_ACTION_LABELS))
        self.observation_space = env.observation_space

    def step(self, action):
        label = HRL_SPEED_ACTION_LABELS[int(np.asarray(action).item())]
        mapped_action = hrl_action_index(self.env, label)
        obs, base_reward, terminated, truncated, info = self.env.step(mapped_action)
        reward = self._speed_reward(label, base_reward)
        info = dict(info)
        info.update(self._speed_info(label, reward))
        return obs, reward, terminated, truncated, info

    def _speed_reward(self, label: str, base_reward: float) -> float:
        ego = self.env.unwrapped.vehicle
        collision = bool(getattr(ego, "crashed", False))
        speed = hrl_forward_speed(ego)
        front_ttc = compute_same_lane_ttc(self.env, ttc_cap=HRL_TTC_CAP)
        local_speed = hrl_local_traffic_speed(self.env)
        speed_deficit = max(0.0, local_speed - speed - 2.0)
        lag_penalty = float(np.clip(speed_deficit / 10.0, 0.0, 1.0))
        low_ttc_penalty = hrl_ttc_low_penalty(front_ttc)
        speed_bonus = hrl_speed_score(speed, self.config)
        action_cost = 0.02 if label != "IDLE" else 0.0
        return float(
            -15.0 * collision
            - 2.5 * low_ttc_penalty
            - 0.8 * lag_penalty
            - action_cost
            + 0.7 * speed_bonus
            + 0.2 * min(float(front_ttc), HRL_TTC_CAP) / HRL_TTC_CAP
        )

    def _speed_info(self, label: str, reward: float) -> dict:
        ego = self.env.unwrapped.vehicle
        front_ttc = compute_same_lane_ttc(self.env, ttc_cap=HRL_TTC_CAP)
        local_speed = hrl_local_traffic_speed(self.env)
        return {
            "hrl_speed_action": label,
            "hrl_speed_reward": float(reward),
            "hrl_front_ttc": float(front_ttc),
            "hrl_local_traffic_speed": float(local_speed),
            "hrl_speed_deficit": float(max(0.0, local_speed - hrl_forward_speed(ego) - 2.0)),
        }


class HRLUpperLaneEnv(gym.Wrapper):
    """Upper option policy: lane-left, lane-right, or delegate to keep-lane speed policy."""

    def __init__(self, env: gym.Env, config: dict, speed_model: DQN):
        super().__init__(env)
        self.config = dict(config)
        self.speed_model = speed_model
        self.action_space = spaces.Discrete(len(HRL_UPPER_ACTION_LABELS))
        self.observation_space = env.observation_space

    def step(self, action):
        option = HRL_UPPER_ACTION_LABELS[int(np.asarray(action).item())]
        if option == "KEEP_LANE":
            speed_action, _ = self.speed_model.predict(self._last_obs, deterministic=True)
            speed_label = HRL_SPEED_ACTION_LABELS[int(np.asarray(speed_action).item())]
            mapped_action = hrl_action_index(self.env, speed_label)
            chosen_lane_context = hrl_lane_context(self.env, lane_delta=0)
        else:
            speed_label = "delegated_only_for_keep_lane"
            lane_delta = -1 if option == "LANE_LEFT" else 1
            chosen_lane_context = hrl_lane_context(self.env, lane_delta=lane_delta)
            mapped_action = hrl_action_index(self.env, option)

        obs, base_reward, terminated, truncated, info = self.env.step(mapped_action)
        self._last_obs = obs
        reward = self._upper_reward(option, chosen_lane_context)
        info = dict(info)
        info.update(self._upper_info(option, speed_label, chosen_lane_context, reward))
        return obs, reward, terminated, truncated, info

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._last_obs = obs
        return obs, info

    def _upper_reward(self, option: str, lane_context: dict) -> float:
        ego = self.env.unwrapped.vehicle
        collision = bool(getattr(ego, "crashed", False))
        speed = hrl_forward_speed(ego)
        front_ttc = float(lane_context["front_ttc"])
        rear_ttc = float(lane_context["rear_ttc"])
        safety_score = min(front_ttc, rear_ttc, HRL_TTC_CAP) / HRL_TTC_CAP
        lane_speed_score = hrl_speed_score(float(lane_context["lane_mean_speed"]), self.config)
        ego_speed_score = hrl_speed_score(speed, self.config)
        lane_change_cost = 0.05 if option != "KEEP_LANE" else 0.0
        invalid_penalty = 2.0 if not bool(lane_context["valid"]) else 0.0
        return float(
            -15.0 * collision
            - invalid_penalty
            - lane_change_cost
            + 0.55 * safety_score
            + 0.35 * lane_speed_score
            + 0.35 * ego_speed_score
        )

    def _upper_info(self, option: str, speed_label: str, lane_context: dict, reward: float) -> dict:
        return {
            "hrl_upper_action": option,
            "hrl_delegated_speed_action": speed_label,
            "hrl_upper_reward": float(reward),
            "hrl_candidate_lane_valid": bool(lane_context["valid"]),
            "hrl_candidate_front_ttc": float(lane_context["front_ttc"]),
            "hrl_candidate_rear_ttc": float(lane_context["rear_ttc"]),
            "hrl_candidate_lane_mean_speed": float(lane_context["lane_mean_speed"]),
        }


class HRLTimestepProgressCallback(BaseCallback):
    def __init__(self, total_timesteps: int, every_n_steps: int = 10_000):
        super().__init__(verbose=0)
        self.total_timesteps = max(1, int(total_timesteps))
        self.every_n_steps = max(1, int(every_n_steps))
        self.next_print = self.every_n_steps

    def _on_step(self) -> bool:
        if self.num_timesteps >= self.next_print:
            progress = min(100.0, 100.0 * self.num_timesteps / self.total_timesteps)
            print(f"[hrl train] {self.num_timesteps}/{self.total_timesteps} timesteps ({progress:.1f}%)", flush=True)
            while self.next_print <= self.num_timesteps:
                self.next_print += self.every_n_steps
        return True


def make_hrl_speed_env(config: dict, seed_value: int, rank: int = 0, monitor_dir=None):
    env = HRLKeepLaneSpeedEnv(hrl_make_base_env(config), config=config)
    env.reset(seed=seed_value + rank)
    if monitor_dir is not None:
        monitor_dir.mkdir(parents=True, exist_ok=True)
        return Monitor(env, filename=str(monitor_dir / f"speed_env_{rank}.monitor.csv"))
    return Monitor(env)


def make_hrl_upper_env(config: dict, speed_model: DQN, seed_value: int, rank: int = 0, monitor_dir=None):
    env = HRLUpperLaneEnv(hrl_make_base_env(config), config=config, speed_model=speed_model)
    env.reset(seed=seed_value + rank)
    if monitor_dir is not None:
        monitor_dir.mkdir(parents=True, exist_ok=True)
        return Monitor(env, filename=str(monitor_dir / f"upper_env_{rank}.monitor.csv"))
    return Monitor(env)


def make_hrl_vec_env(make_one, n_envs_value: int):
    return DummyVecEnv([lambda rank=rank: make_one(rank) for rank in range(int(n_envs_value))])


def train_or_load_hrl_speed_policy(run_dir: Path, config: dict) -> DQN:
    speed_model_path = run_dir / "speed_policy.zip"
    if speed_model_path.exists() and not HRL_FORCE_RETRAIN:
        print("Loading HRL speed policy:", speed_model_path)
        return DQN.load(str(speed_model_path), device="cuda" if torch.cuda.is_available() else "cpu")

    train_env = make_hrl_vec_env(
        lambda rank: make_hrl_speed_env(config, seed + 11_000, rank, run_dir / "speed_monitor"),
        HRL_N_ENVS,
    )
    eval_env = make_hrl_speed_env(config, seed + 21_000)
    callbacks = CallbackList([
        HRLTimestepProgressCallback(HRL_SPEED_TIMESTEPS, every_n_steps=10_000),
        EvalCallback(
            eval_env,
            best_model_save_path=str(run_dir / "speed_best_model"),
            log_path=str(run_dir / "speed_eval_tracker"),
            eval_freq=max(10_000 // max(1, HRL_N_ENVS), 1),
            n_eval_episodes=eval_episodes_during_training,
            deterministic=True,
            render=False,
        ),
    ])
    model = DQN(
        "MlpPolicy",
        train_env,
        seed=seed + 11_000,
        device="cuda" if torch.cuda.is_available() else "cpu",
        **HRL_HYPERPARAMETERS,
    )
    model.learn(total_timesteps=HRL_SPEED_TIMESTEPS, callback=callbacks)
    model.save(speed_model_path)
    train_env.close()
    eval_env.close()
    return model


def train_or_load_hrl_upper_policy(run_dir: Path, config: dict, speed_model: DQN) -> DQN:
    upper_model_path = run_dir / "upper_policy.zip"
    if upper_model_path.exists() and not HRL_FORCE_RETRAIN:
        print("Loading HRL upper policy:", upper_model_path)
        return DQN.load(str(upper_model_path), device="cuda" if torch.cuda.is_available() else "cpu")

    train_env = make_hrl_vec_env(
        lambda rank: make_hrl_upper_env(config, speed_model, seed + 31_000, rank, run_dir / "upper_monitor"),
        HRL_N_ENVS,
    )
    eval_env = make_hrl_upper_env(config, speed_model, seed + 41_000)
    callbacks = CallbackList([
        HRLTimestepProgressCallback(HRL_UPPER_TIMESTEPS, every_n_steps=10_000),
        EvalCallback(
            eval_env,
            best_model_save_path=str(run_dir / "upper_best_model"),
            log_path=str(run_dir / "upper_eval_tracker"),
            eval_freq=max(10_000 // max(1, HRL_N_ENVS), 1),
            n_eval_episodes=eval_episodes_during_training,
            deterministic=True,
            render=False,
        ),
    ])
    model = DQN(
        "MlpPolicy",
        train_env,
        seed=seed + 31_000,
        device="cuda" if torch.cuda.is_available() else "cpu",
        **HRL_HYPERPARAMETERS,
    )
    model.learn(total_timesteps=HRL_UPPER_TIMESTEPS, callback=callbacks)
    model.save(upper_model_path)
    train_env.close()
    eval_env.close()
    return model


def hrl_update_overtake_tracker(env, seen_ahead_ids: set[int], overtaken_ids: set[int]) -> None:
    ego = env.unwrapped.vehicle
    ego_x = float(ego.position[0])
    for other in env.unwrapped.road.vehicles:
        if other is ego:
            continue
        other_id = id(other)
        dx = float(other.position[0] - ego_x)
        if dx > 0.0:
            seen_ahead_ids.add(other_id)
        elif other_id in seen_ahead_ids:
            overtaken_ids.add(other_id)


def evaluate_hrl_policy(upper_model: DQN, speed_model: DQN, config: dict, episodes: int, eval_seed: int) -> pd.DataFrame:
    env = make_hrl_upper_env(config, speed_model, eval_seed)
    rows = []
    try:
        for episode_idx in range(int(episodes)):
            obs, _ = env.reset(seed=eval_seed + episode_idx)
            terminated = truncated = False
            total_reward = 0.0
            steps = 0
            speed_trace = []
            ttc_trace = []
            upper_actions = []
            speed_actions = []
            invalid_lane_actions = 0
            seen_ahead_ids = set()
            overtaken_ids = set()
            final_info = {}
            while not (terminated or truncated):
                hrl_update_overtake_tracker(env, seen_ahead_ids, overtaken_ids)
                speed_trace.append(hrl_forward_speed(env.unwrapped.vehicle))
                ttc_trace.append(compute_same_lane_ttc(env, ttc_cap=HRL_TTC_CAP))
                action, _ = upper_model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = env.step(action)
                final_info = dict(info)
                upper_actions.append(str(info.get("hrl_upper_action", "unknown")))
                speed_actions.append(str(info.get("hrl_delegated_speed_action", "")))
                invalid_lane_actions += int(not bool(info.get("hrl_candidate_lane_valid", True)))
                total_reward += float(reward)
                steps += 1
            rows.append({
                "episode": episode_idx + 1,
                "steps": int(steps),
                "reward": float(total_reward),
                "collision": bool(getattr(env.unwrapped.vehicle, "crashed", False)),
                "avg_speed": float(np.mean(speed_trace)) if speed_trace else 0.0,
                "overtakes": int(len(overtaken_ids)),
                "avg_ttc": float(np.mean(ttc_trace)) if ttc_trace else HRL_TTC_CAP,
                "min_ttc": float(np.min(ttc_trace)) if ttc_trace else HRL_TTC_CAP,
                "keep_lane_rate": float(upper_actions.count("KEEP_LANE") / max(len(upper_actions), 1)),
                "lane_left_rate": float(upper_actions.count("LANE_LEFT") / max(len(upper_actions), 1)),
                "lane_right_rate": float(upper_actions.count("LANE_RIGHT") / max(len(upper_actions), 1)),
                "invalid_lane_action_rate": float(invalid_lane_actions / max(len(upper_actions), 1)),
                "delegated_faster_rate": float(speed_actions.count("FASTER") / max(len(upper_actions), 1)),
                "delegated_slower_rate": float(speed_actions.count("SLOWER") / max(len(upper_actions), 1)),
                "final_upper_action": str(final_info.get("hrl_upper_action", "unknown")),
            })
    finally:
        env.close()
    return pd.DataFrame(rows)


def summarize_hrl_eval(eval_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for column, label, scale in [
        ("avg_speed", "Average Speed (m/s)", 1.0),
        ("collision", "Collision Rate (%)", 100.0),
        ("overtakes", "Overtakes", 1.0),
        ("avg_ttc", "Average TTC (s)", 1.0),
        ("min_ttc", "Minimum TTC (s)", 1.0),
        ("reward", "Reward", 1.0),
        ("keep_lane_rate", "Keep Lane Rate", 1.0),
        ("invalid_lane_action_rate", "Invalid Lane Action Rate", 1.0),
    ]:
        values = pd.to_numeric(eval_df[column], errors="coerce").fillna(0.0).astype(float) * scale
        std = float(values.std(ddof=0))
        row = {"metric": label, "mean": float(values.mean()), "std": std, "standard_error": float(std / np.sqrt(max(len(values), 1)))}
        if column == "collision":
            row["collisions"] = int(pd.to_numeric(eval_df[column], errors="coerce").fillna(0).astype(bool).sum())
            row["episodes"] = int(len(values))
        rows.append(row)
    return pd.DataFrame(rows)


def plot_hrl_eval(eval_df: pd.DataFrame, save_path: Path) -> None:
    episodes = eval_df["episode"].to_numpy()
    collisions = eval_df["collision"].astype(float).to_numpy()
    running_collision_rate = 100.0 * np.cumsum(collisions) / np.arange(1, len(collisions) + 1)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes[0, 0].plot(episodes, eval_df["avg_speed"], color="tab:green")
    axes[0, 0].set_title("Average Speed")
    axes[0, 1].plot(episodes, running_collision_rate, color="crimson")
    axes[0, 1].set_title("Running Collision Rate")
    axes[1, 0].plot(episodes, eval_df["keep_lane_rate"], label="Keep", color="tab:blue")
    axes[1, 0].plot(episodes, eval_df["lane_left_rate"], label="Left", color="tab:orange")
    axes[1, 0].plot(episodes, eval_df["lane_right_rate"], label="Right", color="tab:purple")
    axes[1, 0].set_title("Upper Option Rates")
    axes[1, 0].legend()
    axes[1, 1].plot(episodes, eval_df["min_ttc"], label="Min TTC", color="tab:blue")
    axes[1, 1].plot(episodes, eval_df["avg_ttc"], label="Avg TTC", color="tab:purple")
    axes[1, 1].set_title("Time To Collision")
    axes[1, 1].legend()
    for ax in axes.flat:
        ax.grid(alpha=0.3)
    fig.suptitle(f"{HRL_LABEL} Saved-Model Evaluation")
    fig.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def run_hrl_experiment() -> dict:
    run_dir = RESULTS_DIR / HRL_RUN_NAME
    run_dir.mkdir(parents=True, exist_ok=True)
    env_config = hrl_base_env_config()
    display(pd.DataFrame({"hrl_training_env": pd.Series(env_config)}))
    started = time.time()
    speed_model = train_or_load_hrl_speed_policy(run_dir, env_config)
    upper_model = train_or_load_hrl_upper_policy(run_dir, env_config, speed_model)
    elapsed_seconds = time.time() - started

    eval_df = evaluate_hrl_policy(
        upper_model,
        speed_model,
        env_config,
        episodes=HRL_EVAL_EPISODES,
        eval_seed=saved_model_eval_seed + 900,
    )
    eval_dir = run_dir / saved_model_eval_name
    eval_dir.mkdir(parents=True, exist_ok=True)
    eval_metrics_path = eval_dir / "evaluation_metrics.json"
    eval_summary_path = eval_dir / "evaluation_summary.json"
    eval_plot_path = eval_dir / "evaluation_metrics.png"
    eval_metrics_path.write_text(eval_df.to_json(orient="records", indent=2), encoding="utf-8")
    plot_hrl_eval(eval_df, eval_plot_path)
    metric_summary = summarize_hrl_eval(eval_df)

    summary = {
        "run_name": HRL_RUN_NAME,
        "label": HRL_LABEL,
        "algorithm": "Hierarchical DQN",
        "timesteps": {"speed_policy": int(HRL_SPEED_TIMESTEPS), "upper_policy": int(HRL_UPPER_TIMESTEPS)},
        "elapsed_seconds": float(elapsed_seconds),
        "speed_model_path": str(run_dir / "speed_policy.zip"),
        "upper_model_path": str(run_dir / "upper_policy.zip"),
        "model_path": str(run_dir / "upper_policy.zip"),
        "env_config": env_config,
        "hyperparameters": HRL_HYPERPARAMETERS,
    }
    summary_path = run_dir / "summary.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    saved_eval_summary = {
        "episodes": int(HRL_EVAL_EPISODES),
        "seed": int(saved_model_eval_seed + 900),
        "model_path": str(run_dir / "upper_policy.zip"),
        "speed_model_path": str(run_dir / "speed_policy.zip"),
        "env_config": env_config,
        "evaluation_metrics_path": str(eval_metrics_path),
        "detailed_plot_path": str(eval_plot_path),
        "metric_summary": metric_summary.to_dict(orient="records"),
    }
    eval_summary_path.write_text(json.dumps(saved_eval_summary, indent=2), encoding="utf-8")

    output = {
        "run_name": HRL_RUN_NAME,
        "label": HRL_LABEL,
        "backend_module": "hierarchical_dqn",
        "algorithm": "Hierarchical DQN",
        "attention": False,
        "adaptive": False,
        "safety_reward": True,
        "aggressiveness_state": False,
        "hierarchical": True,
        "summary": summary,
        "saved_eval_summary": saved_eval_summary,
    }
    completed_experiments[HRL_RUN_NAME] = output
    print(json.dumps(summary, indent=2))
    display(metric_summary.round(3))
    display(eval_df.head(10).round(3))
    return output


experiment_9_hierarchical_lane_speed = run_hrl_experiment()


## Render HRL Episodes


In [ ]:
from IPython.display import Image, Video, display

HRL_RENDER_EPISODES = 3
HRL_RENDER_PREVIEW_COUNT = 3
HRL_RENDER_DURATION_MULTIPLIER = 2
HRL_RENDER_SEED = seed + 95_000


def make_long_hrl_render_config(duration_multiplier: int = HRL_RENDER_DURATION_MULTIPLIER) -> dict:
    render_config = hrl_base_env_config()
    render_config["duration"] = int(render_config["duration"] * duration_multiplier)
    render_config["render_duration_multiplier"] = int(duration_multiplier)
    render_config["max_policy_steps"] = int(render_config["duration"] * render_config["policy_frequency"])
    return render_config


def load_saved_hrl_models(run_name: str = HRL_RUN_NAME):
    run_dir = RESULTS_DIR / run_name
    speed_model_path = run_dir / "speed_policy.zip"
    upper_model_path = run_dir / "upper_policy.zip"
    missing = [str(path) for path in [speed_model_path, upper_model_path] if not path.exists()]
    if missing:
        raise FileNotFoundError("Run the HRL experiment first; missing: " + ", ".join(missing))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Loading HRL speed policy:", speed_model_path)
    speed_model = DQN.load(str(speed_model_path), device=device)
    print("Loading HRL upper policy:", upper_model_path)
    upper_model = DQN.load(str(upper_model_path), device=device)
    return speed_model, upper_model


def write_hrl_episode_video(frames: list[np.ndarray], video_path: Path, fps: int) -> Path:
    if not frames:
        raise RuntimeError(f"No frames captured for {video_path}")
    video_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        import cv2
    except ModuleNotFoundError:
        cv2 = None

    first_frame = np.asarray(frames[0])
    height, width = first_frame.shape[:2]
    normalized_frames = []
    for frame in frames:
        frame = np.asarray(frame)
        if frame.shape[:2] != (height, width):
            if cv2 is not None:
                frame = cv2.resize(frame, (width, height))
            else:
                from PIL import Image as PILImage
                frame = np.asarray(PILImage.fromarray(frame).resize((width, height)))
        normalized_frames.append(frame.astype(np.uint8))

    if cv2 is not None:
        writer = cv2.VideoWriter(
            str(video_path),
            cv2.VideoWriter_fourcc(*"mp4v"),
            int(fps),
            (int(width), int(height)),
        )
        try:
            for frame in normalized_frames:
                writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
        finally:
            writer.release()
        return video_path

    try:
        import imageio.v2 as imageio
        imageio.mimsave(video_path, normalized_frames, fps=int(fps), macro_block_size=1)
        return video_path
    except Exception:
        from PIL import Image as PILImage
        gif_path = video_path.with_suffix(".gif")
        duration_ms = int(1000 / max(1, int(fps)))
        pil_frames = [PILImage.fromarray(frame) for frame in normalized_frames]
        pil_frames[0].save(
            gif_path,
            save_all=True,
            append_images=pil_frames[1:],
            duration=duration_ms,
            loop=0,
        )
        print(f"OpenCV/imageio MP4 writing unavailable, saved GIF instead: {gif_path}")
        return gif_path


def render_hrl_episodes(
    *,
    episodes: int = HRL_RENDER_EPISODES,
    seed: int = HRL_RENDER_SEED,
    preview_count: int = HRL_RENDER_PREVIEW_COUNT,
    duration_multiplier: int = HRL_RENDER_DURATION_MULTIPLIER,
) -> pd.DataFrame:
    speed_model, upper_model = load_saved_hrl_models()
    render_config = make_long_hrl_render_config(duration_multiplier=duration_multiplier)
    run_dir = RESULTS_DIR / HRL_RUN_NAME
    render_dir = run_dir / f"render_long_{episodes}_episodes_{render_config['duration']}s"
    render_dir.mkdir(parents=True, exist_ok=True)
    fps = int(render_config.get("policy_frequency", 3))
    rows = []
    env = HRLUpperLaneEnv(
        hrl_make_base_env(render_config, render_mode="rgb_array"),
        config=render_config,
        speed_model=speed_model,
    )
    try:
        for episode_idx in range(int(episodes)):
            obs, _ = env.reset(seed=seed + episode_idx)
            first_frame = env.render()
            frames = [first_frame] if first_frame is not None else []
            terminated = truncated = False
            steps = 0
            total_reward = 0.0
            speed_trace = []
            ttc_trace = []
            upper_actions = []
            invalid_lane_actions = 0
            while not (terminated or truncated) and steps < int(render_config["max_policy_steps"]):
                speed_trace.append(hrl_forward_speed(env.unwrapped.vehicle))
                ttc_trace.append(compute_same_lane_ttc(env, ttc_cap=HRL_TTC_CAP))
                action, _ = upper_model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = env.step(action)
                total_reward += float(reward)
                upper_actions.append(str(info.get("hrl_upper_action", "unknown")))
                invalid_lane_actions += int(not bool(info.get("hrl_lane_action_valid", True)))
                steps += 1
                frame = env.render()
                if frame is not None:
                    frames.append(frame)
            video_path = render_dir / f"episode_{episode_idx + 1:02d}.mp4"
            written_path = write_hrl_episode_video(frames, video_path, fps=fps)
            rows.append(
                {
                    "episode": episode_idx + 1,
                    "seed": seed + episode_idx,
                    "duration_seconds": int(render_config["duration"]),
                    "max_policy_steps": int(render_config["max_policy_steps"]),
                    "steps": int(steps),
                    "reward": float(total_reward),
                    "collision": bool(env.unwrapped.vehicle.crashed),
                    "avg_speed": float(np.mean(speed_trace)) if speed_trace else 0.0,
                    "min_ttc": float(np.min(ttc_trace)) if ttc_trace else HRL_TTC_CAP,
                    "keep_lane_rate": float(upper_actions.count("KEEP_LANE") / max(len(upper_actions), 1)),
                    "lane_left_rate": float(upper_actions.count("LANE_LEFT") / max(len(upper_actions), 1)),
                    "lane_right_rate": float(upper_actions.count("LANE_RIGHT") / max(len(upper_actions), 1)),
                    "invalid_lane_action_rate": float(invalid_lane_actions / max(len(upper_actions), 1)),
                    "video_path": str(written_path),
                }
            )
            print(f"HRL render {episode_idx + 1}/{episodes}: {written_path}")
    finally:
        env.close()

    render_df = pd.DataFrame(rows)
    metrics_path = render_dir / "render_metrics.csv"
    summary_path = render_dir / "render_summary.json"
    render_df.to_csv(metrics_path, index=False)
    summary_path.write_text(
        json.dumps(
            {
                "run_name": HRL_RUN_NAME,
                "label": HRL_LABEL,
                "episodes": int(episodes),
                "seed": int(seed),
                "duration_multiplier": int(duration_multiplier),
                "duration_seconds": int(render_config["duration"]),
                "max_policy_steps": int(render_config["max_policy_steps"]),
                "env_config": render_config,
                "render_dir": str(render_dir),
                "metrics_path": str(metrics_path),
                "videos": render_df["video_path"].tolist(),
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    display(render_df[["episode", "seed", "duration_seconds", "steps", "reward", "collision", "avg_speed", "min_ttc", "keep_lane_rate", "video_path"]].round(3))
    for video_path in render_df["video_path"].head(int(preview_count)):
        if str(video_path).lower().endswith(".mp4"):
            display(Video(video_path, embed=False, html_attributes="controls muted"))
        else:
            display(Image(filename=video_path))
    return render_df


hrl_render_df = render_hrl_episodes()


## Experiment 10-11: SAC Continuous Target Policies


In [ ]:
import time
from functools import partial

import gymnasium as gym
import highway_env  # noqa: F401 - register highway-v0
import matplotlib.pyplot as plt
import numpy as np
import torch
from gymnasium import spaces
from highway_env import utils
from highway_env.envs.common.action import ActionType
from highway_env.envs.common.observation import observation_factory
from highway_env.envs.highway_env import HighwayEnv
from highway_env.vehicle.controller import ControlledVehicle
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import BaseCallback, CallbackList, EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

DQN_SCRIPT_DIR = PROJECT_ROOT / "src" / "deep_learning" / "DQN"
dqn_script_dir_str = str(DQN_SCRIPT_DIR)
if dqn_script_dir_str not in sys.path:
    sys.path.insert(0, dqn_script_dir_str)

from adaptive_longitudinal import DriverAggressivenessWrapper, TTCObservationWrapper

SAC_TIMESTEPS = timesteps
SAC_EVAL_EVERY_TIMESTEPS = 10_000
SAC_TRAINING_EVAL_EPISODES = eval_episodes_during_training
SAC_N_ENVS = n_envs
SAC_POLICY_KWARGS = {"net_arch": [256, 256]}
SAC_TENSORBOARD_ROOT = Path(os.environ.get("CONGESTED_SAC_TB_ROOT", r"C:\\tmp\\agv_sac_tb"))
SAC_TENSORBOARD_ROOT.mkdir(parents=True, exist_ok=True)
SAC_HYPERPARAMETERS = {
    "learning_rate": 3e-4,
    "buffer_size": 200_000,
    "learning_starts": 10_000,
    "batch_size": 256,
    "gamma": 0.995,
    "tau": 0.005,
    "train_freq": 1,
    "gradient_steps": 1,
    "ent_coef": "auto",
    "policy_kwargs": SAC_POLICY_KWARGS,
}


class CongestedSACTargetVehicle(ControlledVehicle):
    """Ego vehicle controlled by target speed and continuous road-lateral target."""

    ACCELERATION_LIMIT = 5.0

    def __init__(
        self,
        *args,
        lane_width: float = 4.0,
        lanes_count: int = 3,
        target_boundary_margin: float = 1.8,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.lane_width = float(lane_width)
        self.lanes_count = int(lanes_count)
        self.target_boundary_margin = float(target_boundary_margin)
        self.road_center_y = 0.5 * (self.lanes_count - 1) * self.lane_width
        self.target_y = float(self.position[1])
        self.last_lateral_command = 0.0

    def lateral_bounds(self) -> tuple[float, float]:
        low = -0.5 * self.lane_width + self.target_boundary_margin
        high = (self.lanes_count - 0.5) * self.lane_width - self.target_boundary_margin
        return float(low), float(high)

    def act(self, action=None) -> None:
        if self.crashed:
            return

        self.follow_road()
        if action:
            if "target_speed_delta" in action:
                self.target_speed += float(action["target_speed_delta"])
            if "target_speed" in action:
                self.target_speed = float(action["target_speed"])
            if "target_y_delta" in action:
                self.target_y += float(action["target_y_delta"])
            if "target_y" in action:
                self.target_y = float(action["target_y"])

        low_y, high_y = self.lateral_bounds()
        self.target_y = float(np.clip(self.target_y, low_y, high_y))
        self.target_speed = float(np.clip(self.target_speed, 0.0, self.MAX_SPEED))
        self.target_lane_index = self._nearest_lane_index(self.target_y)

        low_level_action = {
            "acceleration": self.speed_control(self.target_speed),
            "steering": self.steering_to_lateral_target(self.target_y),
        }
        low_level_action["acceleration"] = float(np.clip(low_level_action["acceleration"], -self.ACCELERATION_LIMIT, self.ACCELERATION_LIMIT))
        low_level_action["steering"] = float(np.clip(low_level_action["steering"], -self.MAX_STEERING_ANGLE, self.MAX_STEERING_ANGLE))
        super(ControlledVehicle, self).act(low_level_action)

    def _nearest_lane_index(self, target_y: float):
        lane_from, lane_to, _ = self.target_lane_index
        lane_count = len(self.road.network.graph[lane_from][lane_to])
        lane_id = int(np.clip(np.round(float(target_y) / self.lane_width), 0, lane_count - 1))
        return (lane_from, lane_to, lane_id)

    def steering_to_lateral_target(self, target_y: float) -> float:
        lane = self.road.network.get_lane(self.lane_index)
        longitudinal, _ = lane.local_coordinates(self.position)
        lane_heading = lane.heading_at(longitudinal + self.speed * self.TAU_PURSUIT)
        lateral_error = float(self.position[1] - target_y)
        lateral_speed_command = -self.KP_LATERAL * lateral_error
        heading_command = np.arcsin(np.clip(lateral_speed_command / max(abs(self.speed), 1e-6), -1.0, 1.0))
        heading_ref = lane_heading + np.clip(heading_command, -np.pi / 4, np.pi / 4)
        heading_rate_command = self.KP_HEADING * (heading_ref - self.heading)
        slip_angle = np.arcsin(np.clip(self.LENGTH / 2 / max(abs(self.speed), 1e-6) * heading_rate_command, -1.0, 1.0))
        return float(np.arctan(2 * np.tan(slip_angle)))


class CongestedContinuousTargetAction(ActionType):
    """Continuous SAC action: [speed_delta, lateral_delta], both normalized to [-1, 1]."""

    def __init__(
        self,
        env,
        speed_delta: float = 3.0,
        max_lateral_delta: float = 1.5,
        lane_width: float = 4.0,
        target_boundary_margin: float = 1.8,
        **kwargs,
    ):
        super().__init__(env)
        self.speed_delta = float(speed_delta)
        self.max_lateral_delta = float(max_lateral_delta)
        self.lane_width = float(lane_width)
        self.target_boundary_margin = float(target_boundary_margin)

    def space(self):
        return spaces.Box(-1.0, 1.0, shape=(2,), dtype=np.float32)

    @property
    def vehicle_class(self):
        return partial(
            CongestedSACTargetVehicle,
            lane_width=self.lane_width,
            lanes_count=self.env.config.get("lanes_count", 3),
            target_boundary_margin=self.target_boundary_margin,
        )

    def act(self, action: np.ndarray) -> None:
        action = np.clip(np.asarray(action, dtype=np.float32), -1.0, 1.0)
        self.controlled_vehicle.last_lateral_command = float(action[1])
        self.controlled_vehicle.act(
            {
                "target_speed_delta": float(action[0]) * self.speed_delta,
                "target_y_delta": float(action[1]) * self.max_lateral_delta,
            }
        )


class CongestedSACLanelessRewardMixin:
    @classmethod
    def default_config(cls) -> dict:
        config = super().default_config()
        config.update(
            {
                "lane_width": 4.0,
                "ttc_penalty": -0.6,
                "ttc_threshold": 4.0,
                "ttc_lateral_margin": 0.5,
                "traffic_lag_penalty": -0.3,
                "traffic_lag_speed_tolerance": 2.0,
                "traffic_lag_max_speed_error": 10.0,
                "local_traffic_radius": 120.0,
                "target_boundary_margin": 1.8,
                "boundary_penalty": -0.35,
                "middle_position_reward": 0.30,
                "target_lateral_penalty": -0.08,
                "lateral_action_penalty": -0.04,
                "normalize_shaping_reward": True,
                "spatial_shaping_reward_weight": 0.35,
                "safety_flow_shaping_reward_weight": 0.35,
            }
        )
        return config

    def _front_ttc(self) -> float:
        ego = self.vehicle
        ego_x, ego_y = float(ego.position[0]), float(ego.position[1])
        ego_vx = ego.speed * np.cos(ego.heading)
        min_ttc = np.inf
        for other in self.road.vehicles:
            if other is ego:
                continue
            dx = float(other.position[0] - ego_x)
            if dx <= 0.0:
                continue
            lateral_overlap = abs(float(other.position[1]) - ego_y) <= (
                ego.WIDTH / 2 + other.WIDTH / 2 + self.config["ttc_lateral_margin"]
            )
            if not lateral_overlap:
                continue
            other_vx = other.speed * np.cos(other.heading)
            closing_speed = ego_vx - other_vx
            if closing_speed > 1e-6:
                min_ttc = min(min_ttc, dx / closing_speed)
        return float(min_ttc)

    def _ttc_safety_reward(self) -> float:
        min_ttc = self._front_ttc()
        if not np.isfinite(min_ttc):
            return 1.0
        return float(np.clip(min_ttc / self.config["ttc_threshold"], 0.0, 1.0))

    def _traffic_flow_reward(self) -> float:
        ego = self.vehicle
        ego_x = float(ego.position[0])
        nearby_speeds = [
            float(other.speed)
            for other in self.road.vehicles
            if other is not ego and abs(float(other.position[0]) - ego_x) <= self.config["local_traffic_radius"]
        ]
        if not nearby_speeds:
            return 1.0
        local_mean_speed = float(np.mean(nearby_speeds))
        lag = max(0.0, local_mean_speed - ego.speed - self.config["traffic_lag_speed_tolerance"])
        normalized_lag = np.clip(lag / self.config["traffic_lag_max_speed_error"], 0.0, 1.0)
        return float(1.0 - normalized_lag)

    def _road_center_y(self) -> float:
        return 0.5 * (self.config["lanes_count"] - 1) * self.config["lane_width"]

    def _usable_half_width(self) -> float:
        road_width = self.config["lanes_count"] * self.config["lane_width"]
        return float(max(1e-6, 0.5 * road_width - self.config["target_boundary_margin"]))

    def _normalized_lateral_distance(self) -> float:
        centered_y = float(self.vehicle.position[1]) - self._road_center_y()
        return float(np.clip(abs(centered_y) / self._usable_half_width(), 0.0, 1.0))

    def _boundary_margin_reward(self) -> float:
        normalized_distance = self._normalized_lateral_distance()
        return float(1.0 - normalized_distance**2)

    def _middle_position_reward(self) -> float:
        normalized_distance = self._normalized_lateral_distance()
        return float(1.0 - normalized_distance)

    def _target_lateral_penalty(self) -> float:
        target_y = float(getattr(self.vehicle, "target_y", self.vehicle.position[1]))
        centered_target_y = target_y - self._road_center_y()
        return float(np.clip(abs(centered_target_y) / self._usable_half_width(), 0.0, 1.0))

    def _lateral_action_penalty(self) -> float:
        return float(np.clip(abs(getattr(self.vehicle, "last_lateral_command", 0.0)), 0.0, 1.0))

    def _spatial_shaping_components(self, rewards: dict[str, float]) -> dict[str, float]:
        return {
            "boundary_penalty": self.config["boundary_penalty"] * (1.0 - rewards["boundary_margin_reward"]),
            "middle_position_reward": self.config["middle_position_reward"] * rewards["middle_position_reward"],
            "target_lateral_penalty": self.config["target_lateral_penalty"] * rewards["target_lateral_penalty"],
            "lateral_action_penalty": self.config["lateral_action_penalty"] * rewards["lateral_action_penalty"],
        }

    def _safety_flow_shaping_components(self, rewards: dict[str, float]) -> dict[str, float]:
        return {
            "ttc_penalty": self.config["ttc_penalty"] * (1.0 - rewards["ttc_safety_reward"]),
            "traffic_lag_penalty": self.config["traffic_lag_penalty"] * (1.0 - rewards["traffic_flow_reward"]),
        }

    def _normalize_component_sum(self, components: dict[str, float]) -> float:
        raw_shaping = float(sum(components.values()))
        if not self.config.get("normalize_shaping_reward", True):
            return raw_shaping
        positive_scale = max(1e-6, sum(max(0.0, float(self.config[key])) for key in components))
        negative_scale = max(1e-6, sum(abs(min(0.0, float(self.config[key]))) for key in components))
        normalized = raw_shaping / (positive_scale if raw_shaping >= 0.0 else negative_scale)
        return float(np.clip(normalized, -1.0, 1.0))

    def _normalized_shaping_reward(self, rewards: dict[str, float]) -> float:
        spatial = self._normalize_component_sum(self._spatial_shaping_components(rewards))
        safety_flow = self._normalize_component_sum(self._safety_flow_shaping_components(rewards))
        return float(
            self.config["spatial_shaping_reward_weight"] * spatial
            + self.config["safety_flow_shaping_reward_weight"] * safety_flow
        )

    def _rewards(self, action) -> dict[str, float]:
        rewards = super()._rewards(action)
        rewards["ttc_safety_reward"] = self._ttc_safety_reward()
        rewards["traffic_flow_reward"] = self._traffic_flow_reward()
        rewards["boundary_margin_reward"] = self._boundary_margin_reward()
        rewards["middle_position_reward"] = self._middle_position_reward()
        rewards["target_lateral_penalty"] = self._target_lateral_penalty()
        rewards["lateral_action_penalty"] = self._lateral_action_penalty()
        rewards["normalized_shaping_reward"] = self._normalized_shaping_reward(rewards)
        return rewards

    def _reward(self, action) -> float:
        base_rewards = HighwayEnv._rewards(self, action)
        reward = sum(
            self.config.get(name, 0) * reward_value
            for name, reward_value in base_rewards.items()
        )
        if self.config["normalize_reward"]:
            reward = utils.lmap(
                reward,
                [
                    self.config["collision_reward"],
                    self.config["high_speed_reward"] + self.config["right_lane_reward"],
                ],
                [0, 1],
            )
        reward *= base_rewards["on_road_reward"]
        rewards = self._rewards(action)
        reward += rewards["normalized_shaping_reward"]
        return float(reward)


class CongestedSACEnv(CongestedSACLanelessRewardMixin, HighwayEnv):
    def define_spaces(self) -> None:
        self.observation_type = observation_factory(self, self.config["observation"])
        action_cfg = dict(self.config["action"])
        if action_cfg.get("type") == "CongestedContinuousTargetAction":
            action_cfg.pop("type", None)
            self.action_type = CongestedContinuousTargetAction(self, **action_cfg)
        else:
            super().define_spaces()
            return
        self.observation_space = self.observation_type.space()
        self.action_space = self.action_type.space()


def make_congested_sac_env_config(*, safety_weighted: bool = False) -> dict:
    lane_width = 4.0
    cfg = build_env_config(
        profile_name=environment_profile,
        profile_overrides=congested_environment_overrides,
        observation=congested_observation_config,
        action={
            "type": "CongestedContinuousTargetAction",
            "speed_delta": 3.0,
            "max_lateral_delta": 1.5,
            "lane_width": lane_width,
            "target_boundary_margin": 1.8,
        },
        reward=base_reward_config,
        speed=speed_config,
        adaptive_longitudinal={"enabled": False},
        rear_flow={"enabled": False},
        traffic_flow_reward={"enabled": False},
        safety_ttc_flow_reward={"enabled": False},
        driver_aggressiveness=congested_driver_aggressiveness_config,
        ttc_observation=congested_ttc_observation_config,
    )
    cfg.update(
        {
            "lane_width": lane_width,
            "ttc_penalty": -0.6,
            "ttc_threshold": 4.0,
            "ttc_lateral_margin": 0.5,
            "traffic_lag_penalty": -0.3,
            "traffic_lag_speed_tolerance": 2.0,
            "traffic_lag_max_speed_error": 10.0,
            "local_traffic_radius": 120.0,
            "target_boundary_margin": 1.8,
            "boundary_penalty": -0.35,
            "middle_position_reward": 0.30,
            "target_lateral_penalty": -0.08,
            "lateral_action_penalty": -0.04,
            "normalize_shaping_reward": True,
            "spatial_shaping_reward_weight": 0.35,
            "safety_flow_shaping_reward_weight": 0.35,
        }
    )
    if safety_weighted:
        cfg.update(
            {
                "ttc_penalty": -1.0,
                "traffic_lag_penalty": -0.35,
                "spatial_shaping_reward_weight": 0.30,
                "safety_flow_shaping_reward_weight": 0.50,
            }
        )
    return cfg


def make_congested_sac_env(render_mode=None, config=None):
    env_config = dict(config or make_congested_sac_env_config())
    env = CongestedSACEnv(config=env_config, render_mode=render_mode)
    if env_config.get("driver_aggressiveness", {}).get("enabled", False):
        env = DriverAggressivenessWrapper(env, driver_aggressiveness_config=env_config["driver_aggressiveness"])
    if env_config.get("ttc_observation", {}).get("enabled", False):
        env = TTCObservationWrapper(env, ttc_observation_config=env_config["ttc_observation"])
    return env


class SACTimestepProgressCallback(BaseCallback):
    def __init__(self, total_timesteps: int, every_n_steps: int = 10_000):
        super().__init__(verbose=0)
        self.total_timesteps = max(1, int(total_timesteps))
        self.every_n_steps = max(1, int(every_n_steps))
        self.next_print = self.every_n_steps

    def _on_step(self) -> bool:
        if self.num_timesteps >= self.next_print:
            progress = min(100.0, 100.0 * self.num_timesteps / self.total_timesteps)
            print(f"[sac train] {self.num_timesteps}/{self.total_timesteps} timesteps ({progress:.1f}%)", flush=True)
            while self.next_print <= self.num_timesteps:
                self.next_print += self.every_n_steps
        return True


def make_vec_congested_sac_env(config: dict, seed: int, monitor_dir: Path, n_envs: int = SAC_N_ENVS):
    monitor_dir.mkdir(parents=True, exist_ok=True)

    def make_one(rank: int):
        def _init():
            env = make_congested_sac_env(config=config, render_mode=None)
            env.reset(seed=seed + rank)
            return Monitor(env, filename=str(monitor_dir / f"env_{rank}.monitor.csv"))
        return _init

    return DummyVecEnv([make_one(rank) for rank in range(int(n_envs))])


def make_single_congested_sac_env(config: dict, seed: int):
    env = make_congested_sac_env(config=config, render_mode=None)
    env.reset(seed=seed)
    return Monitor(env)


def compute_congested_sac_ttc(env, ttc_cap: float = 10.0) -> float:
    ttc = env.unwrapped._front_ttc() if hasattr(env.unwrapped, "_front_ttc") else np.inf
    if not np.isfinite(ttc):
        return float(ttc_cap)
    return float(np.clip(ttc, 0.0, ttc_cap))


def update_congested_overtake_tracker(env, seen_ahead_ids: set[int], overtaken_ids: set[int]) -> None:
    ego = env.unwrapped.vehicle
    ego_x = float(ego.position[0])
    for other in env.unwrapped.road.vehicles:
        if other is ego:
            continue
        other_id = id(other)
        dx = float(other.position[0] - ego_x)
        if dx > 0.0:
            seen_ahead_ids.add(other_id)
        elif other_id in seen_ahead_ids:
            overtaken_ids.add(other_id)


def evaluate_congested_sac_model(model, config: dict, episodes: int, seed: int, ttc_cap: float = 10.0):
    env = make_single_congested_sac_env(config, seed=seed)
    rows = []
    try:
        for episode_idx in range(int(episodes)):
            obs, _ = env.reset(seed=seed + episode_idx)
            terminated = truncated = False
            total_reward = 0.0
            steps = 0
            speed_trace = []
            ttc_trace = []
            shaping_trace = []
            middle_trace = []
            seen_ahead_ids = set()
            overtaken_ids = set()
            while not (terminated or truncated):
                update_congested_overtake_tracker(env, seen_ahead_ids, overtaken_ids)
                speed_trace.append(float(env.unwrapped.vehicle.speed))
                ttc_trace.append(compute_congested_sac_ttc(env, ttc_cap=ttc_cap))
                action, _ = model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = env.step(action)
                total_reward += float(reward)
                rewards = info.get("rewards", {})
                if "normalized_shaping_reward" in rewards:
                    shaping_trace.append(float(rewards["normalized_shaping_reward"]))
                if "middle_position_reward" in rewards:
                    middle_trace.append(float(rewards["middle_position_reward"]))
                steps += 1
            rows.append(
                {
                    "episode": episode_idx + 1,
                    "steps": steps,
                    "reward": total_reward,
                    "collision": bool(env.unwrapped.vehicle.crashed),
                    "avg_speed": float(np.mean(speed_trace)) if speed_trace else 0.0,
                    "overtakes": int(len(overtaken_ids)),
                    "avg_ttc": float(np.mean(ttc_trace)) if ttc_trace else float(ttc_cap),
                    "min_ttc": float(np.min(ttc_trace)) if ttc_trace else float(ttc_cap),
                    "normalized_shaping_reward": float(np.mean(shaping_trace)) if shaping_trace else np.nan,
                    "middle_position_reward": float(np.mean(middle_trace)) if middle_trace else np.nan,
                }
            )
    finally:
        env.close()
    return rows


def summarize_congested_sac_eval(eval_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for column, label, scale in [
        ("avg_speed", "Average Speed (m/s)", 1.0),
        ("collision", "Collision Rate (%)", 100.0),
        ("overtakes", "Overtakes", 1.0),
        ("avg_ttc", "Average TTC (s)", 1.0),
        ("min_ttc", "Minimum TTC (s)", 1.0),
        ("reward", "Reward", 1.0),
        ("normalized_shaping_reward", "Normalized Shaping Reward", 1.0),
    ]:
        if column not in eval_df:
            continue
        values = pd.to_numeric(eval_df[column], errors="coerce").fillna(0.0).astype(float) * scale
        rows.append({"metric": label, "mean": float(values.mean()), "std": float(values.std(ddof=0))})
    return pd.DataFrame(rows)


def plot_congested_sac_evaluation(eval_df: pd.DataFrame, save_path: Path, title: str) -> None:
    episodes = eval_df["episode"].to_numpy()
    collisions = eval_df["collision"].astype(float).to_numpy()
    running_collision_rate = 100.0 * np.cumsum(collisions) / np.arange(1, len(collisions) + 1)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes[0, 0].plot(episodes, eval_df["avg_speed"], color="tab:green")
    axes[0, 0].set_title("Average Speed")
    axes[0, 1].plot(episodes, running_collision_rate, color="crimson")
    axes[0, 1].set_title("Running Collision Rate")
    axes[1, 0].bar(episodes, eval_df["overtakes"], color="tab:orange")
    axes[1, 0].set_title("Overtakes")
    axes[1, 1].plot(episodes, eval_df["min_ttc"], label="Min TTC", color="tab:blue")
    axes[1, 1].plot(episodes, eval_df["avg_ttc"], label="Avg TTC", color="tab:purple")
    axes[1, 1].set_title("Time To Collision")
    axes[1, 1].legend()
    for ax in axes.flat:
        ax.grid(alpha=0.3)
    fig.suptitle(title)
    fig.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def train_congested_sac_experiment(*, run_name: str, label: str, safety_weighted: bool = False, seed_offset: int = 0) -> dict:
    run_dir = RESULTS_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    env_config = make_congested_sac_env_config(safety_weighted=safety_weighted)
    train_env = make_vec_congested_sac_env(env_config, seed=seed + seed_offset, monitor_dir=run_dir / "monitor", n_envs=SAC_N_ENVS)
    eval_env = make_single_congested_sac_env(env_config, seed=seed + seed_offset + 10_000)
    tensorboard_run_name = f"sac_{seed_offset}"
    callbacks = CallbackList(
        [
            SACTimestepProgressCallback(SAC_TIMESTEPS, every_n_steps=SAC_EVAL_EVERY_TIMESTEPS),
            EvalCallback(
                eval_env,
                best_model_save_path=str(run_dir / "best_model"),
                log_path=str(run_dir / "eval_tracker"),
                eval_freq=max(SAC_EVAL_EVERY_TIMESTEPS // max(1, int(SAC_N_ENVS)), 1),
                n_eval_episodes=SAC_TRAINING_EVAL_EPISODES,
                deterministic=True,
                render=False,
            ),
        ]
    )
    model = SAC(
        "MlpPolicy",
        train_env,
        seed=seed + seed_offset,
        device="cuda" if torch.cuda.is_available() else "cpu",
        tensorboard_log=str(SAC_TENSORBOARD_ROOT),
        verbose=1,
        **SAC_HYPERPARAMETERS,
    )
    started = time.time()
    model.learn(total_timesteps=SAC_TIMESTEPS, callback=callbacks, tb_log_name=tensorboard_run_name)
    elapsed_seconds = time.time() - started
    model_path = run_dir / "model.zip"
    model.save(model_path)
    train_env.close()
    eval_env.close()

    eval_rows = evaluate_congested_sac_model(
        model,
        env_config,
        episodes=saved_model_eval_episodes,
        seed=saved_model_eval_seed + seed_offset,
    )
    eval_df = pd.DataFrame(eval_rows)
    eval_dir = run_dir / saved_model_eval_name
    eval_dir.mkdir(parents=True, exist_ok=True)
    eval_metrics_path = eval_dir / "evaluation_metrics.json"
    eval_plot_path = eval_dir / "evaluation_metrics.png"
    eval_summary_path = eval_dir / "evaluation_summary.json"
    eval_metrics_path.write_text(eval_df.to_json(orient="records", indent=2), encoding="utf-8")
    plot_congested_sac_evaluation(eval_df, eval_plot_path, f"{label} Saved-Model Evaluation")
    metric_summary = summarize_congested_sac_eval(eval_df)
    saved_eval_summary = {
        "episodes": int(saved_model_eval_episodes),
        "seed": int(saved_model_eval_seed + seed_offset),
        "model_path": str(model_path),
        "env_config": env_config,
        "evaluation_metrics_path": str(eval_metrics_path),
        "detailed_plot_path": str(eval_plot_path),
        "metric_summary": metric_summary.to_dict(orient="records"),
    }
    eval_summary_path.write_text(json.dumps(saved_eval_summary, indent=2), encoding="utf-8")

    summary = {
        "run_name": run_name,
        "label": label,
        "algorithm": "SAC",
        "timesteps": int(SAC_TIMESTEPS),
        "elapsed_seconds": float(elapsed_seconds),
        "model_path": str(model_path),
        "env_config": env_config,
        "hyperparameters": SAC_HYPERPARAMETERS,
        "tensorboard_log_root": str(SAC_TENSORBOARD_ROOT),
        "tensorboard_run_name": tensorboard_run_name,
    }
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    output = {
        "run_name": run_name,
        "label": label,
        "backend_module": "sac_continuous_target",
        "algorithm": "SAC",
        "safety_reward": bool(safety_weighted),
        "adaptive": False,
        "attention": False,
        "summary": summary,
        "saved_eval_summary": saved_eval_summary,
    }
    completed_experiments[run_name] = output
    print(json.dumps(summary, indent=2))
    display(metric_summary.round(3))
    display(eval_df.head(10).round(3))
    return output


sac_env_smoke = make_congested_sac_env(config=make_congested_sac_env_config(), render_mode=None)
sac_smoke_obs, _ = sac_env_smoke.reset(seed=seed + 70_000)
sac_smoke_next_obs, sac_smoke_reward, sac_smoke_terminated, sac_smoke_truncated, sac_smoke_info = sac_env_smoke.step(sac_env_smoke.action_space.sample())
print("Congested SAC observation space:", sac_env_smoke.observation_space)
print("Congested SAC action space:", sac_env_smoke.action_space)
print("Congested SAC sample reward:", sac_smoke_reward)
print("Congested SAC reward terms:", sac_smoke_info.get("rewards", {}))
sac_env_smoke.close()

experiment_8_sac_laneless_reward = train_congested_sac_experiment(
    run_name="congested_sac_laneless_reward_v2_100k",
    label="Congested Traffic - SAC Continuous Target + Laneless Reward",
    safety_weighted=False,
    seed_offset=700,
)

experiment_9_sac_laneless_reward_safety_weighted = train_congested_sac_experiment(
    run_name="congested_sac_laneless_reward_safety_weighted_v2_100k",
    label="Congested Traffic - SAC Continuous Target + Safety-Weighted Laneless Reward",
    safety_weighted=True,
    seed_offset=800,
)


## Render SAC With Longer Episodes


In [ ]:
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, Video, display
from stable_baselines3 import SAC

SAC_RENDER_EPISODES = 10
SAC_RENDER_PREVIEW_COUNT = 10
SAC_RENDER_DURATION_MULTIPLIER = 3
SAC_RENDER_SEED = seed + 90_000
SAC_RENDER_RUNS = [
    {
        "run_name": "congested_sac_laneless_reward_v2_100k",
        "label": "Congested SAC + Laneless Reward",
        "safety_weighted": False,
        "seed_offset": 0,
    },
    {
        "run_name": "congested_sac_laneless_reward_safety_weighted_v2_100k",
        "label": "Congested SAC + Safety-Weighted Laneless Reward",
        "safety_weighted": True,
        "seed_offset": 1_000,
    },
]


def make_long_sac_render_config(*, safety_weighted: bool = False, duration_multiplier: int = SAC_RENDER_DURATION_MULTIPLIER) -> dict:
    render_config = make_congested_sac_env_config(safety_weighted=safety_weighted)
    render_config["duration"] = int(render_config["duration"] * duration_multiplier)
    render_config["render_duration_multiplier"] = int(duration_multiplier)
    render_config["max_policy_steps"] = int(render_config["duration"] * render_config["policy_frequency"])
    return render_config


def load_saved_congested_sac_model(run_name: str):
    run_dir = RESULTS_DIR / run_name
    summary_path = run_dir / "summary.json"
    candidates = []
    if summary_path.exists():
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        model_path = Path(summary.get("model_path", ""))
        if model_path:
            candidates.append(model_path)
    candidates.extend([run_dir / "model.zip", run_dir / "best_model" / "best_model.zip"])
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            print(f"Loading SAC model {run_name} from {candidate}")
            return SAC.load(str(candidate), device="cuda" if torch.cuda.is_available() else "cpu")
    raise FileNotFoundError(f"No saved SAC model found for {run_name} in {run_dir}")


def write_sac_episode_video(frames: list[np.ndarray], video_path: Path, fps: int) -> Path:
    if not frames:
        raise RuntimeError(f"No frames captured for {video_path}")
    video_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        import cv2
    except ModuleNotFoundError:
        cv2 = None

    first_frame = np.asarray(frames[0])
    height, width = first_frame.shape[:2]
    normalized_frames = []
    for frame in frames:
        frame = np.asarray(frame)
        if frame.shape[:2] != (height, width):
            if cv2 is not None:
                frame = cv2.resize(frame, (width, height))
            else:
                from PIL import Image as PILImage
                frame = np.asarray(PILImage.fromarray(frame).resize((width, height)))
        normalized_frames.append(frame.astype(np.uint8))

    if cv2 is not None:
        writer = cv2.VideoWriter(
            str(video_path),
            cv2.VideoWriter_fourcc(*"mp4v"),
            int(fps),
            (int(width), int(height)),
        )
        try:
            for frame in normalized_frames:
                writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
        finally:
            writer.release()
        return video_path

    try:
        import imageio.v2 as imageio
        imageio.mimsave(video_path, normalized_frames, fps=int(fps), macro_block_size=1)
        return video_path
    except Exception:
        from PIL import Image as PILImage
        gif_path = video_path.with_suffix(".gif")
        duration_ms = int(1000 / max(1, int(fps)))
        pil_frames = [PILImage.fromarray(frame) for frame in normalized_frames]
        pil_frames[0].save(
            gif_path,
            save_all=True,
            append_images=pil_frames[1:],
            duration=duration_ms,
            loop=0,
        )
        print(f"OpenCV/imageio MP4 writing unavailable, saved GIF instead: {gif_path}")
        return gif_path


def render_congested_sac_episodes(
    *,
    run_name: str,
    label: str,
    safety_weighted: bool = False,
    episodes: int = SAC_RENDER_EPISODES,
    seed: int = SAC_RENDER_SEED,
    preview_count: int = SAC_RENDER_PREVIEW_COUNT,
    duration_multiplier: int = SAC_RENDER_DURATION_MULTIPLIER,
) -> pd.DataFrame:
    model = load_saved_congested_sac_model(run_name)
    render_config = make_long_sac_render_config(
        safety_weighted=safety_weighted,
        duration_multiplier=duration_multiplier,
    )
    run_dir = RESULTS_DIR / run_name
    render_dir = run_dir / f"render_long_{episodes}_episodes_{render_config['duration']}s"
    render_dir.mkdir(parents=True, exist_ok=True)
    fps = int(render_config.get("policy_frequency", 3))
    rows = []
    env = make_congested_sac_env(config=render_config, render_mode="rgb_array")
    try:
        for episode_idx in range(int(episodes)):
            obs, _ = env.reset(seed=seed + episode_idx)
            first_frame = env.render()
            frames = [first_frame] if first_frame is not None else []
            terminated = truncated = False
            total_reward = 0.0
            steps = 0
            speed_trace = []
            ttc_trace = []
            shaping_trace = []
            while not (terminated or truncated):
                speed_trace.append(float(env.unwrapped.vehicle.speed))
                ttc_trace.append(compute_congested_sac_ttc(env))
                action, _ = model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = env.step(action)
                total_reward += float(reward)
                rewards = info.get("rewards", {})
                if "normalized_shaping_reward" in rewards:
                    shaping_trace.append(float(rewards["normalized_shaping_reward"]))
                steps += 1
                frame = env.render()
                if frame is not None:
                    frames.append(frame)
            video_path = render_dir / f"episode_{episode_idx + 1:02d}.mp4"
            written_path = write_sac_episode_video(frames, video_path, fps=fps)
            rows.append(
                {
                    "episode": episode_idx + 1,
                    "seed": seed + episode_idx,
                    "duration_seconds": render_config["duration"],
                    "max_policy_steps": render_config["max_policy_steps"],
                    "steps": int(steps),
                    "reward": float(total_reward),
                    "collision": bool(env.unwrapped.vehicle.crashed),
                    "avg_speed": float(np.mean(speed_trace)) if speed_trace else 0.0,
                    "min_ttc": float(np.min(ttc_trace)) if ttc_trace else 10.0,
                    "avg_normalized_shaping_reward": float(np.mean(shaping_trace)) if shaping_trace else np.nan,
                    "video_path": str(written_path),
                }
            )
            print(f"{label} long render {episode_idx + 1}/{episodes}: {written_path}")
    finally:
        env.close()

    render_df = pd.DataFrame(rows)
    metrics_path = render_dir / "render_metrics.csv"
    summary_path = render_dir / "render_summary.json"
    render_df.to_csv(metrics_path, index=False)
    summary_path.write_text(
        json.dumps(
            {
                "label": label,
                "run_name": run_name,
                "episodes": int(episodes),
                "seed": int(seed),
                "duration_multiplier": int(duration_multiplier),
                "duration_seconds": int(render_config["duration"]),
                "max_policy_steps": int(render_config["max_policy_steps"]),
                "env_config": render_config,
                "render_dir": str(render_dir),
                "metrics_path": str(metrics_path),
                "videos": render_df["video_path"].tolist(),
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    display(render_df[["episode", "seed", "duration_seconds", "steps", "reward", "collision", "avg_speed", "min_ttc", "video_path"]].round(3))
    for video_path in render_df["video_path"].head(int(preview_count)):
        if str(video_path).lower().endswith(".mp4"):
            display(Video(video_path, embed=False, html_attributes="controls muted"))
        else:
            display(Image(filename=video_path))
    return render_df


sac_long_render_outputs = {}
for render_spec in SAC_RENDER_RUNS:
    try:
        sac_long_render_outputs[render_spec["run_name"]] = render_congested_sac_episodes(
            run_name=render_spec["run_name"],
            label=render_spec["label"],
            safety_weighted=bool(render_spec.get("safety_weighted", False)),
            episodes=SAC_RENDER_EPISODES,
            seed=SAC_RENDER_SEED + int(render_spec.get("seed_offset", 0)),
            preview_count=SAC_RENDER_PREVIEW_COUNT,
            duration_multiplier=SAC_RENDER_DURATION_MULTIPLIER,
        )
    except FileNotFoundError as exc:
        print(f"Skipping long SAC render: {exc}")


## Collision Diagnostics

Run saved congested-traffic models through action- and lane-level diagnostics. This does not retrain models; it loads existing `summary.json` / `model_path` artifacts and labels each episode by likely failure mode.

In [1]:
from stable_baselines3 import DQN

DQN_SCRIPT_DIR = PROJECT_ROOT / "src" / "deep_learning" / "DQN"
dqn_script_dir_str = str(DQN_SCRIPT_DIR)
if dqn_script_dir_str not in sys.path:
    sys.path.insert(0, dqn_script_dir_str)

from congestion_diagnostics import (
    evaluate_congestion_diagnostics,
    save_congestion_diagnostics,
)

CONGESTED_RUN_SPECS = [
    {"run_name": "congested_lane_safety_baseline_dqn_v2_100k", "label": "Baseline DQN + Lane Context/Safety", "backend_module": "elurant_dqn", "seed_offset": 0},
    {"run_name": "congested_lane_safety_attention_dqn_v2_100k", "label": "Attention DQN + Lane Context/Safety", "backend_module": "attention_dqn", "attention": True, "seed_offset": 100},
    {"run_name": "congested_lane_safety_attention_safety_reward_v2_100k", "label": "Attention DQN + Lane Context/Safety + TTC Reward Safety", "backend_module": "attention_dqn", "attention": True, "safety_reward": True, "seed_offset": 200},
    {"run_name": "congested_lane_safety_adaptive_attention_safety_reward_v2_100k", "label": "Adaptive Attention DQN + Lane Context/Safety + TTC Reward Safety", "backend_module": "attention_dqn", "attention": True, "adaptive": True, "safety_reward": True, "seed_offset": 300},
]

DIAGNOSTIC_EPISODES = 100
DIAGNOSTIC_SEED = seed + 50_000
DIAGNOSTIC_CONFIG = {
    "front_ttc_safe": 4.0,
    "front_ttc_critical": 1.5,
    "rear_ttc_safe": 4.0,
    "rear_ttc_critical": 1.5,
    "lane_gap_safe": 12.0,
    "bad_action_margin": 0.35,
    "no_good_action_risk": 0.85,
}


def diagnostic_failure_reason(row: pd.Series) -> str:
    if not bool(row.get("collision", False)):
        return "no_collision"
    if bool(row.get("agent_chose_badly", False)):
        return "bad_final_action"
    if bool(row.get("wrong_lane_earlier", False)):
        return "missed_better_lane"
    if bool(row.get("unavoidable_rear_pressure", False)):
        return "rear_pressure_no_good_action"
    if bool(row.get("no_good_discrete_action", False)):
        return "no_good_discrete_action"
    return f"collision_{row.get('collision_type', 'unknown')}"


def run_collision_diagnostics_for_spec(spec: dict, episodes: int = DIAGNOSTIC_EPISODES):
    run_name = spec["run_name"]
    run_dir = RESULTS_DIR / run_name
    summary_path = run_dir / "summary.json"
    if not summary_path.exists():
        print(f"Skipping {run_name}: missing {summary_path}")
        return None

    trainer, _, _, _, default_device = load_dqn_backend(
        backend_module=spec["backend_module"],
        notebook_subdir="congested_traffic_policy",
        results_subdir=RESULTS_SUBDIR,
    )
    run_summary = json.loads(summary_path.read_text(encoding="utf-8"))
    model_path = Path(run_summary["model_path"])
    if not model_path.exists():
        print(f"Skipping {run_name}: missing model {model_path}")
        return None

    env_config = make_congested_env_config(
        safety_reward=bool(spec.get("safety_reward", False)),
        adaptive=bool(spec.get("adaptive", False)),
        aggressiveness_state=bool(spec.get("aggressiveness_state", False)),
    )
    print(f"Running collision diagnostics for {run_name} ({episodes} episodes)")
    model = DQN.load(str(model_path), device=default_device)
    diagnostic_df, diagnostic_traces = evaluate_congestion_diagnostics(
        model,
        trainer.make_env,
        env_config=env_config,
        episodes=episodes,
        seed=DIAGNOSTIC_SEED + int(spec.get("seed_offset", 0)),
        diagnostic_config=DIAGNOSTIC_CONFIG,
    )
    diagnostic_df["run_name"] = run_name
    diagnostic_df["label"] = spec["label"]
    diagnostic_df["failure_reason"] = diagnostic_df.apply(diagnostic_failure_reason, axis=1)

    output_dir = run_dir / f"collision_diagnostics_{episodes}_episodes"
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = save_congestion_diagnostics(diagnostic_df, diagnostic_traces, output_dir)
    enriched_path = output_dir / "collision_diagnostic_summary_enriched.csv"
    diagnostic_df.to_csv(enriched_path, index=False)

    failure_counts = (
        diagnostic_df["failure_reason"]
        .value_counts(dropna=False)
        .rename_axis("failure_reason")
        .reset_index(name="episodes")
    )
    failure_counts["rate"] = failure_counts["episodes"] / max(len(diagnostic_df), 1)
    failure_counts.to_csv(output_dir / "failure_reason_counts.csv", index=False)

    return {
        "run_name": run_name,
        "label": spec["label"],
        "episodes": int(episodes),
        "collision_rate": float(diagnostic_df["collision"].mean()) if len(diagnostic_df) else 0.0,
        "bad_final_action_rate": float(diagnostic_df["agent_chose_badly"].mean()) if len(diagnostic_df) else 0.0,
        "no_good_action_rate": float(diagnostic_df["no_good_discrete_action"].mean()) if len(diagnostic_df) else 0.0,
        "missed_better_lane_rate": float(diagnostic_df["wrong_lane_earlier"].mean()) if len(diagnostic_df) else 0.0,
        "unavoidable_rear_pressure_rate": float(diagnostic_df["unavoidable_rear_pressure"].mean()) if len(diagnostic_df) else 0.0,
        "mean_bad_action_count": float(diagnostic_df["bad_action_count"].mean()) if len(diagnostic_df) else 0.0,
        "mean_missed_better_lane_count": float(diagnostic_df["missed_better_lane_count"].mean()) if len(diagnostic_df) else 0.0,
        "summary_path": paths["summary_path"],
        "traces_path": paths["traces_path"],
        "enriched_csv_path": str(enriched_path),
        "failure_counts_path": str(output_dir / "failure_reason_counts.csv"),
    }


ModuleNotFoundError: No module named 'stable_baselines3'

In [ ]:
diagnostic_outputs = []
for spec in CONGESTED_RUN_SPECS:
    output = run_collision_diagnostics_for_spec(spec, episodes=DIAGNOSTIC_EPISODES)
    if output is not None:
        diagnostic_outputs.append(output)

collision_diagnostic_comparison_df = pd.DataFrame(diagnostic_outputs)
comparison_diagnostic_path = RESULTS_DIR / f"collision_diagnostic_comparison_{DIAGNOSTIC_EPISODES}_episodes.csv"
collision_diagnostic_comparison_df.to_csv(comparison_diagnostic_path, index=False)
print("Collision diagnostic comparison saved to:", comparison_diagnostic_path)
display(collision_diagnostic_comparison_df.round(3))


In [ ]:
# Failure-mode matrix across all diagnostic runs.
failure_frames = []
for output in diagnostic_outputs:
    counts_path = Path(output["failure_counts_path"])
    if counts_path.exists():
        counts = pd.read_csv(counts_path)
        counts["run_name"] = output["run_name"]
        counts["label"] = output["label"]
        failure_frames.append(counts)

if failure_frames:
    failure_reason_df = pd.concat(failure_frames, ignore_index=True)
    failure_reason_matrix = (
        failure_reason_df.pivot_table(
            index=["run_name", "label"],
            columns="failure_reason",
            values="rate",
            fill_value=0.0,
        )
        .reset_index()
    )
    matrix_path = RESULTS_DIR / f"collision_failure_reason_matrix_{DIAGNOSTIC_EPISODES}_episodes.csv"
    failure_reason_matrix.to_csv(matrix_path, index=False)
    print("Failure reason matrix saved to:", matrix_path)
    display(failure_reason_matrix.round(3))
else:
    print("No diagnostic failure-count files found. Run the diagnostics cell first.")


## Comparison Table

In [ ]:
# Aggregate the saved 1k evaluation summaries for all runs that have completed.
rows = []
for run_name, output in completed_experiments.items():
    summary_records = output["saved_eval_summary"].get("metric_summary", [])
    row = {
        "run_name": run_name,
        "label": output["label"],
        "algorithm": output.get("algorithm", "DQN"),
        "attention": bool(output.get("attention", False)),
        "adaptive": bool(output.get("adaptive", False)),
        "safety_reward": bool(output.get("safety_reward", False)),
        "lane_change_safety": bool(output.get("lane_change_safety", False)),
        "aggressiveness_state": bool(output.get("aggressiveness_state", False)),
        "hierarchical": bool(output.get("hierarchical", False)),
    }
    for record in summary_records:
        metric = record.get("metric", "").replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_per_").lower()
        row[f"{metric}_mean"] = record.get("mean")
        row[f"{metric}_std"] = record.get("std")
    rows.append(row)

comparison_df = pd.DataFrame(rows)
comparison_path = RESULTS_DIR / "congested_policy_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print("Comparison saved to:", comparison_path)
display(comparison_df.round(3))

In [ ]:
# Learned Behaviour Analysis: baseline vs attention vs safety reward vs adaptive safety reward.
# Uses saved 1k evaluation files for aggregate plots, then runs deterministic matched-seed replays for action-level behaviour.
from IPython.display import Image

notebook_dir_str = str(NOTEBOOK_DIR)
if notebook_dir_str not in sys.path:
    sys.path.insert(0, notebook_dir_str)

import learned_behaviour_analysis
learned_behaviour_analysis = importlib.reload(learned_behaviour_analysis)

LEARNED_BEHAVIOUR_REPLAY_EPISODES = 200
LEARNED_BEHAVIOUR_FORCE_REPLAY = False

learned_behaviour_outputs = learned_behaviour_analysis.run_learned_behaviour_analysis(
    project_root=PROJECT_ROOT,
    results_dir=RESULTS_DIR,
    results_subdir=RESULTS_SUBDIR,
    make_congested_env_config=make_congested_env_config,
    load_dqn_backend=load_dqn_backend,
    seed=seed,
    replay_episodes=LEARNED_BEHAVIOUR_REPLAY_EPISODES,
    force_replay=LEARNED_BEHAVIOUR_FORCE_REPLAY,
)

print("Learned-behaviour analysis saved to:", learned_behaviour_outputs["output_dir"])
display(learned_behaviour_outputs["metric_summary"].round(3))
if "replay_summary" in learned_behaviour_outputs:
    display(learned_behaviour_outputs["replay_summary"].round(3))
    print("Matched trace episode:", learned_behaviour_outputs["matched_trace_episode"])

for plot_path in learned_behaviour_outputs["plot_paths"]:
    print("Saved plot:", plot_path)
    display(Image(filename=str(plot_path)))